# Update `Listed Country` from ISO Alpha‑2 codes

This notebook defines a function that:

1. Takes a **source CSV** path (your dataset) and a **lookup CSV** path (the complete ISO file we created).
2. Looks at the `Listed Country` column. If a cell contains **exactly two uppercase letters** (e.g., `US`, `AU`, `VN`), it replaces it with the **Country (Official)** name from the lookup file.
3. Returns a summary of **how many rows were updated** and optionally writes a new CSV.

It is encoding‑safe (UTF‑8 by default) and won’t change rows that don’t match a 2‑letter code or codes not found in the lookup.

In [10]:
import pandas as pd
from typing import Optional, Tuple

def update_listed_country_from_bbg_with_composite(
    source_csv_path: str,
    exchange_map_csv_path: str,   # must have: BBG_Code, Composite_Code, Country (Friendly)
    output_csv_path: str,
    updated_rows_csv_path: Optional[str] = None,
    source_encoding: str = "utf-8",
    output_encoding: str = "utf-8-sig"
) -> Tuple[pd.DataFrame, int]:
    """Replace 2-character tokens in 'Listed Country' using Bloomberg mapping.
       Stage 1: BBG_Code; Stage 2: Composite_Code fallback.
       Preserves literal 'NA'; normalizes whitespace & casing.
    """

    # ---- Helper: read CSV without turning 'NA' into NaN
    def safe_read_csv(path, **kw):
        return pd.read_csv(
            path,
            dtype=str,
            keep_default_na=False,  # keep 'NA' as 'NA'
            na_values=[],           # disable additional NA tokens
            **kw
        )

    # Load source
    df = safe_read_csv(source_csv_path, encoding=source_encoding)
    if "Listed Country" not in df.columns:
        raise ValueError("Source CSV must contain a 'Listed Country' column.")

    # Normalize source tokens (kill NBSP, trim, uppercase)
    tokens = (df["Listed Country"].astype("string")
                                 .str.replace("\u00A0", " ", regex=False)
                                 .str.strip()
                                 .str.upper())

    # Exactly 2 characters (letters or digits): e.g., US, LN, NA, C1, BB
    mask_two_chars = tokens.str.fullmatch(r"[A-Z0-9]{2}", na=False)

    # Load mapping
    ex = safe_read_csv(exchange_map_csv_path, encoding=source_encoding)
    required = {"BBG_Code", "Composite_Code", "Country (Friendly)"}
    missing = required - set(ex.columns)
    if missing:
        raise ValueError(f"Exchange map CSV missing required columns: {missing}")

    # Normalize keys
    for col in ["BBG_Code", "Composite_Code"]:
        ex[col] = (ex[col].astype("string")
                           .str.replace("\u00A0"," ", regex=False)
                           .str.strip()
                           .str.upper())
    ex["Country (Friendly)"] = ex["Country (Friendly)"].astype("string")

    # Build lookup dicts
    map_bbg  = dict(zip(ex["BBG_Code"],       ex["Country (Friendly)"]))
    map_comp = dict(zip(ex["Composite_Code"], ex["Country (Friendly)"]))

    # Optional sanity: ensure NA exists in mapping
    if "NA" not in map_bbg and "NA" not in map_comp:
        print("⚠️  Mapping file lacks code 'NA'. Rebuild your mapping CSV from Excel using keep_default_na=False.")

    # Stage 1: BBG_Code
    stage1 = tokens.where(mask_two_chars).map(map_bbg)

    # Stage 2: Composite_Code fallback (only where stage1 failed)
    need_stage2 = stage1.isna().reindex(df.index, fill_value=False) & mask_two_chars
    stage2 = tokens.where(need_stage2).map(map_comp)

    # Combine results
    mapped = stage1.reindex(df.index)
    mapped = mapped.where(mapped.notna(), stage2)

    # Rows to update
    to_update_mask = mask_two_chars & mapped.notna()
    updated_count = int(to_update_mask.sum())

    # Export only updated rows (optional)
    if updated_rows_csv_path and updated_count > 0:
        updated_rows = df.loc[to_update_mask].copy()
        updated_rows["_Listed Country (before)"] = df.loc[to_update_mask, "Listed Country"].astype("string")
        updated_rows["_Listed Country (after)"]  = mapped.loc[to_update_mask].astype("string").values
        updated_rows.to_csv(updated_rows_csv_path, index=False, encoding=output_encoding, na_rep="")

    # Apply updates
    df.loc[to_update_mask, "Listed Country"] = mapped.loc[to_update_mask].to_numpy()

    # Keep blanks blank (no 'nan' strings)
    df["Listed Country"] = df["Listed Country"].where(df["Listed Country"].notna(), "")

    # Save full dataset
    df.to_csv(output_csv_path, index=False, encoding=output_encoding, na_rep="")

    # --- EXTRA: print rows where Stock ID is exactly "N/A" or "n/a" (no NaN, no trim/case fold) ---
    if "Stock ID" in df.columns:
        sid_raw = df["Stock ID"].astype("string")  # keep as-is; exact match only
        mask_sid_na_exact = sid_raw.eq("N/A") | sid_raw.eq("n/a")
        count_sid_na_exact = int(mask_sid_na_exact.sum())
        print(f'Rows with Stock ID exactly "N/A" or "n/a": {count_sid_na_exact}')
        if count_sid_na_exact:
            cols_pref = [c for c in [
                "Effective Date","Fund Name","Option Name","Asset Class Name",
                "Name/Kind of Investment Item","Stock ID","Listed Country",
                "Security Identifier","Value (AUD)"
            ] if c in df.columns]
            preview = df.loc[mask_sid_na_exact, cols_pref or df.columns].head(50)
            print(preview.to_string(index=False))
    # --------------------------------------------------------------------

    # Tiny unmapped summary to help debug (what 2-char tokens still didn't map?)
    unmapped = tokens[mask_two_chars & ~to_update_mask]
    if not unmapped.empty:
        print("Unmapped 2-char tokens (top 20):")
        print(unmapped.value_counts().head(20).to_string())

    return df, updated_count


# ---- run it immediately with your paths ----
df_updated, changed = update_listed_country_from_bbg_with_composite(
    source_csv_path=r"D:\LinhDao\Programming\SUPERFUNdProject\CareSuper_Cleaned_final.csv",
    exchange_map_csv_path=r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\bloomberg-exchange-codes-full.csv",
    output_csv_path=r"D:\LinhDao\Programming\SUPERFUNdProject\CareSuper_Cleaned_ListedCountry-update.csv",
    updated_rows_csv_path=r"D:\LinhDao\Programming\SUPERFUNdProject\caresuper-updatedrows.csv"
)

print(f"Rows updated (BBG first, Composite fallback): {changed}")


Rows with Stock ID exactly "N/A" or "n/a": 1
Effective Date  Fund Name Option Name Asset Class Name Name/Kind of Investment Item Stock ID Listed Country Value (AUD)
    31/12/2024 Care Super    Balanced    Listed Equity  Realindex Global Share Fund      N/A                845131902.0
Unmapped 2-char tokens (top 20):
Listed Country
S4    1
Rows updated (BBG first, Composite fallback): 2117


In [ ]:
# df_updated, changed = update_listed_country_from_bbg_with_composite(
#     source_csv_path=r"D:\LinhDao\Programming\SUPERFUNdProject\CareSuper_Cleaned_final.csv",
#     exchange_map_csv_path=r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\bloomberg-exchange-codes-full.csv",
#     output_csv_path=r"D:\LinhDao\Programming\SUPERFUNdProject\CareSuper_Cleaned_ListedCountry-update.csv",
#     updated_rows_csv_path=r"D:\LinhDao\Programming\SUPERFUNdProject\caresuper-updatedrows.csv"
# )
# print(f"Rows updated (BBG first, Composite fallback): {changed}")


Rows updated (BBG first, Composite fallback): 2095
